In [1]:
%%writefile requirements.txt
transformers
datasets
accelerate
omegaconf
torchtyping
ipywidgets
matplotlib

Writing requirements.txt


In [2]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 872.5 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.5/159.5 kB 4.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 7.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 23.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.2/305.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 17.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pylab as plt
from omegaconf import DictConfig

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import Dataset, load_dataset

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    set_seed,
)

In [4]:
args = {
    "seed": 42,
    'model_name_or_path': 'facebook/opt-350m',
    'learning_rate': 5e-5,
    'batch_size': 2,
    'gradient_accumulation_steps': 16,
    'num_train_epochs': 1,
    'num_workers': 10,
    'seq_length': 1024,
    'logging_steps': 10,
}

args = DictConfig(args)

set_seed(args.seed)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [6]:
# Response-only Format

def create_custom_dataset(raw_dataset):
    df = raw_dataset.to_pandas()
    negative_df = df[df['label']==0]
    positive_df = df[df['label']==1]
    negative_df = negative_df.drop(
        columns=['label']).rename(
        columns={'text': 'rejected'})
    # shuffle the data
    positive_df = positive_df.sample(
        frac=1, random_state=0).reset_index(
        drop=True).drop(columns=['label']).rename(
        columns={'text': 'chosen'})
    joined_df = negative_df.join(positive_df)

    def tokenize_fn(texts, max_length=args.seq_length):
        encoded = tokenizer(
            texts,
            padding='max_length',
            max_length=max_length,
            truncation=True,
            add_special_tokens=False,
        )
        return encoded

    rejected_encoded = tokenize_fn(joined_df.rejected.values.tolist())
    joined_df['rejected_input_ids'] = rejected_encoded['input_ids']
    joined_df['rejected_attention_mask'] = rejected_encoded['attention_mask']
    encoded_chosen = tokenize_fn(joined_df.chosen.values.tolist())
    joined_df['chosen_input_ids'] = encoded_chosen['input_ids']
    joined_df['chosen_attention_mask'] = encoded_chosen['attention_mask']

    train_dataset = Dataset.from_pandas(joined_df, preserve_index=False)

    return train_dataset.with_format("torch")

In [7]:
raw_dataset = load_dataset("imdb")
del raw_dataset['unsupervised']

train_dataset = create_custom_dataset(raw_dataset['train'])

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=args.batch_size)

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [16]:
train_dataset

Dataset({
    features: ['rejected', 'chosen', 'rejected_input_ids', 'rejected_attention_mask', 'chosen_input_ids', 'chosen_attention_mask'],
    num_rows: 12500
})

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    args.model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    num_labels=1,
)

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

Some weights of OPTForSequenceClassification were not initialized from the model checkpoint at facebook/opt-350m and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
epoch = 1
print_interval=args.logging_steps
num_batches = len(train_dataloader)
progress_bar = tqdm(total=num_batches*args.num_train_epochs, leave=True)
progress_bar.set_description(f"| Train: Epoch {epoch}, evaluating ... |")
losses = []
temp_losses = []
i = 0


optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate)


for epoch in range(1, args.num_train_epochs+1):

    for batch in train_dataloader:

        chosen_input_ids = batch['chosen_input_ids'].to(model.device)
        chosen_attention_mask = batch['chosen_attention_mask'].to(model.device)
        rejected_input_ids = batch['rejected_input_ids'].to(model.device)
        rejected_attention_mask = batch['rejected_attention_mask'].to(model.device)

        r_w = model(chosen_input_ids, attention_mask=chosen_attention_mask).logits
        r_l = model(rejected_input_ids, attention_mask=rejected_attention_mask).logits

        loss = -F.logsigmoid(r_w - r_l).mean()

        # Accumulate the gradients
        loss /= args.gradient_accumulation_steps
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        if (i + 1) % args.gradient_accumulation_steps == 0 or i + 1 == len(train_dataloader):
            optimizer.step()
            optimizer.zero_grad()

        temp_losses.append( loss.item() )
        if i%print_interval==0:
            progress_bar.set_description(f"| Train: Epoch {epoch}, loss = {loss.item():4f} |")
            progress_bar.refresh()
            losses.append( np.mean(temp_losses) )
            temp_losses = []
        progress_bar.update()
        i+=1


progress_bar.set_description(f"| Train: Epoch {epoch}, loss = {loss.item():4f} |")
progress_bar.refresh()

  0%|          | 0/6250 [00:00<?, ?it/s]

True